# Is the hyperparameter search worth moving to Colab?

At E = 1000 the searches are **81 of the 134 core-hours** in the DNN plan — 60% of
the work. They are one-shot per configuration, need no resume machinery, and each
produces a single small hardware-independent trials pickle. So they are the clean
piece to move off the local machine, *if* a GPU is actually faster at this
workload.

Local reference, measured, seconds per hyperopt evaluation at 4 threads on the
Ryzen 7 7800X3D:

| configuration | s / evaluation |
|---|---|
| own | 25.0 |
| wide | 26.8 |
| joint | 33.1 |

**The decision rule** (handover §11.2): if the GPU is 5× or better, move the
searches and the local machine does only the ~53 backtest core-hours. If it is
1.2×, drop the idea. In between, weigh it against the extra moving parts.

### Do not expect a large speedup

This workload is not compute-bound. The networks are small (≈1,100 training rows,
one or two dense layers), the batch is 192, and `DNNModel.fit` is a hand-written
per-epoch Python loop that calls `model.fit(epochs=1)`, then `model.evaluate` and
`model.predict` on the validation set — three separate Keras entry points per
epoch. Most of the time is per-call dispatch overhead, and a faster device does
not remove it. A profile of one DNN-own/DK1 recalibration puts **60% of the fit
in the two validation passes alone**.

The honest expectation is somewhere between 1× and 3×, with the joint
configuration — 1969 inputs, 168 outputs, the widest matrices — the most likely
to benefit. Measure it; don't assume it.

### What is timed

One call to `dnn_dk1.hyperopt._objective` (own) or `_multizone_objective`
(wide/joint): the same function the real search calls, on the same 363-day
pre-test window, with the same data. Not a proxy for an evaluation — the
evaluation.

Per-evaluation time varies by an order of magnitude across architectures, so
timing *one* evaluation measures the draw and not the device. This notebook
therefore **replays actual trials from the finished E=300 searches**. Every trial
in those pickles records the seconds it took *on the laptop*, so each replayed
evaluation is paired against its own local time — same architecture, same data,
same window — and no second local run is needed to compare.

Seconds per *epoch* is recorded too, because the same architecture can early-stop
at a different epoch on two devices (the float arithmetic differs), and seconds
per evaluation would then be measuring epoch count as much as speed.

## 1. Setup

**Runtime → Change runtime type → GPU**, before running anything.

In [ ]:
!git clone https://github.com/Freddy5445/EPF_Masters.git
%cd EPF_Masters

In [ ]:
import colab_setup

# Installs what Colab lacks (hyperopt, statsmodels), wires the vendored
# epftoolbox from the local tree, mounts Drive, reports the device.
has_gpu = colab_setup.bootstrap()

> **Do not import `dnn_dk1`, `keras` or `tensorflow` in this notebook.**
> TensorFlow grabs the whole GPU on first use, and the benchmark runs in
> subprocesses that would then fail to allocate. Everything below stays in
> `subprocess`; the notebook process only ever touches `pandas`, `json` and `os`.

## 2. Data and trials files

`datasets/` and `experiments/` are gitignored, so the clone has neither. Two
things have to be on Drive:

1. **`nordic_baltic_clean_hourly_local.parquet`** — what `data_cleaning_v2.ipynb`
   writes. The next cell projects every zone in Z into the epftoolbox CSV layout
   inside the runtime, with the same `run_lear_from_clean.py` projection the LEAR
   runs used. `own` needs DK1 only; `wide` and `joint` need all seven.
2. **The three trials files** from the finished E=300 searches, copied from
   `experiments\hyperparameters\` into `<DRIVE>/experiments/hyperparameters/`.
   They are 180–320 KB each. These are what supplies both the architectures to
   replay and the local seconds to compare against:

   ```
   DNN_hyperparameters_nl2_datDK1_clean_load-wind-solar_YT2_SF_CW4_1
   DNN_hyperparameters_nl2_datdnnwide_DK1_YT2_SF_CW4_1
   DNN_hyperparameters_nl2_datdnnjoint_YT2_SF_CW4_1
   ```

   They are read only. Nothing here writes to that folder.

In [ ]:
import os, subprocess, sys, json, glob
import pandas as pd

DRIVE    = '/content/drive/MyDrive/EPF_Masters'
PANEL    = os.path.join(DRIVE, 'nordic_baltic_clean_hourly_local.parquet')
DATASETS = os.path.join(DRIVE, 'datasets')
OUT      = os.path.join(DRIVE, 'experiments')
os.makedirs(DATASETS, exist_ok=True); os.makedirs(OUT, exist_ok=True)

# Mirrors dnn_dk1/zones.py::ZONE_EXOG -- NO2/SE3/SE4 have no usable solar series.
# Hardcoded here only so this cell does not have to import dnn_dk1 (which imports
# keras, which takes the GPU). Verified against the built CSVs below.
ZONE_LAYOUT = {'DK1': 'load-wind-solar', 'DK2': 'load-wind-solar',
               'DE_LU': 'load-wind-solar', 'NL': 'load-wind-solar',
               'NO2': 'load-wind', 'SE3': 'load-wind', 'SE4': 'load-wind'}

if not os.path.exists(PANEL):
    print('No parquet at', PANEL)
    print('Put nordic_baltic_clean_hourly_local.parquet there (Drive web or the '
          'desktop client -- not files.upload()).')
else:
    for zone, exog in ZONE_LAYOUT.items():
        target = os.path.join(DATASETS, f'{zone}_clean_{exog}.csv')
        if os.path.exists(target):
            continue
        subprocess.run([sys.executable, 'run_lear_from_clean.py', '--panel', PANEL,
                        '--zone', zone, '--exog', exog, '--datasets-dir', DATASETS,
                        '--csv-only'], check=True)

for zone, exog in ZONE_LAYOUT.items():
    path = os.path.join(DATASETS, f'{zone}_clean_{exog}.csv')
    if os.path.exists(path):
        head = pd.read_csv(path, index_col=0, nrows=1)
        expected = 1 + len(exog.split('-'))
        flag = 'ok' if head.shape[1] == expected else f'!! expected {expected}'
        print(f'{zone:6s} {head.shape[1]} columns  {flag}')
    else:
        print(f'{zone:6s} MISSING')

HYPER = os.path.join(OUT, 'hyperparameters')
TRIALS = {
    'own':   'DNN_hyperparameters_nl2_datDK1_clean_load-wind-solar_YT2_SF_CW4_1',
    'wide':  'DNN_hyperparameters_nl2_datdnnwide_DK1_YT2_SF_CW4_1',
    'joint': 'DNN_hyperparameters_nl2_datdnnjoint_YT2_SF_CW4_1',
}
print()
for cfg, name in TRIALS.items():
    p = os.path.join(HYPER, name)
    print(f'{cfg:6s} trials file: ' + ('ok' if os.path.exists(p) else f'MISSING at {p}'))

## 3. The benchmark script

Written into the clone so it can be run as a subprocess — once per device, because TensorFlow decides what it can see at import time and CPU-only has to be arranged before that.

In [ ]:
%%writefile bench_hyperopt_device.py
"""Time hyperopt evaluations for one configuration on one device.

Run once per device in a *fresh process*: TensorFlow decides what it can see at
import time, so CPU-only has to be arranged before anything imports it.

    python bench_hyperopt_device.py --config own  --device gpu --draws 6
    python bench_hyperopt_device.py --config own  --device cpu --draws 6

What is timed is one call to ``dnn_dk1.hyperopt._objective`` (own) or
``_multizone_objective`` (wide/joint) -- the same function the real search
calls, on the same window, with the same data. Not a proxy for an evaluation:
the evaluation.

Per-evaluation time varies by an order of magnitude across architectures -- a
1000-neuron net that early-stops at 300 epochs against a 60-neuron one that stops
at 40 -- so timing *one* evaluation measures the draw, not the device. The
architectures to time are therefore fixed in advance, and there are two ways to
fix them:

``--replay <trials file>`` (preferred) replays actual trials from a completed
search. Each trial in those pickles records the seconds it took **on the local
machine**, so this pairs Colab against the laptop trial for trial, on the search's
own distribution of architectures, with no local re-run needed.

Otherwise draws are sampled from the search space with a fixed seed. Still paired
across devices, but drawn from the prior rather than from what TPE actually
explored -- uniform draws include many degenerate architectures that early-stop
almost immediately, so their mean cost understates a real search's.

Nothing is written to the real hyperparameters folder: the objective's own
checkpoint goes to a scratch file, and the trials object is thrown away.
"""
import argparse
import json
import os
import sys
import time

# --- everything that must precede `import tensorflow` --------------------
_pre = argparse.ArgumentParser(add_help=False)
_pre.add_argument("--device", default="gpu", choices=["gpu", "cpu"])
_pre.add_argument("--threads", type=int, default=0,
                  help="intra/inter-op threads; 0 = leave TensorFlow's default")
_known, _ = _pre.parse_known_args()

if _known.device == "cpu":
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
if _known.threads:
    os.environ["OMP_NUM_THREADS"] = str(_known.threads)
    os.environ["TF_NUM_INTRAOP_THREADS"] = str(_known.threads)
    os.environ["TF_NUM_INTEROP_THREADS"] = str(_known.threads)
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np                                          # noqa: E402
import pandas as pd                                         # noqa: E402
from hyperopt import Trials                                 # noqa: E402
from hyperopt.pyll.stochastic import sample                 # noqa: E402

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(HERE, "epftoolbox"))

import tensorflow as tf                                     # noqa: E402
from dnn_dk1 import hyperopt as H                           # noqa: E402
from dnn_dk1 import zones as Z                              # noqa: E402
from dnn_dk1.model import DNNModel                          # noqa: E402

# 363 days back from the last day before the test period -- run_dnn_dk1's
# `hyperopt_window`, restated so this script does not depend on argparse
# defaults there. assert_search_window_precedes_test re-checks it anyway.
HYPEROPT_DAYS = 363


# --- record how many epochs each fit actually ran ------------------------
# Seconds per evaluation confounds device speed with epoch count: the same draw
# can early-stop at a different epoch on two devices, because the float
# arithmetic differs. Seconds per epoch is the quantity that transfers.
_EPOCHS = []


class _CountingDNNModel(DNNModel):
    def fit(self, trainX, trainY, valX, valY):
        self._n_epochs = 0
        original = self._obtain_metrics

        def counted(X, Y):
            self._n_epochs += 1
            return original(X, Y)

        self._obtain_metrics = counted
        try:
            super().fit(trainX, trainY, valX, valY)
        finally:
            _EPOCHS.append(self._n_epochs)


H.DNNModel = _CountingDNNModel   # what _objective / _multizone_objective build


def _sample_draws(space, seed, n):
    """``n`` draws from ``space``, reproducibly.

    hyperopt's pyll sampler wants whatever numpy RNG the installed version was
    written against: releases up to 0.2.x call ``rng.randint`` (legacy
    ``RandomState``), later ones ``rng.integers`` (``Generator``). Both are
    tried rather than pinned, so the same seed gives the same draws on the
    laptop and on Colab whichever is installed. Fails loudly if neither works --
    silently falling back to an unseeded RNG would unpair the comparison, which
    is the one thing this script exists to guarantee.
    """
    errors = []
    for factory in (lambda: np.random.default_rng(seed),
                    lambda: np.random.RandomState(seed)):
        rng = factory()
        try:
            return [sample(space, rng=rng) for _ in range(n)]
        except (AttributeError, TypeError) as exc:
            errors.append(f"{type(rng).__name__}: {exc}")
    raise RuntimeError("hyperopt's sampler accepted neither numpy RNG type: "
                       + "; ".join(errors))


def _replay_draws(space, path, seed, n):
    """``n`` hyperparameter dicts replayed from a completed search.

    Returns ``(draws, local_seconds)``. ``local_seconds[i]`` is what that exact
    evaluation cost on the machine that ran the search -- both objectives record
    it in the trial result -- so the comparison needs no second local run.

    The sample is drawn without replacement at a fixed seed, so it is the same
    set of trials on every device, and its mean is an unbiased estimate of the
    search's mean evaluation cost (evenly-spaced ranks would over-weight the
    tails).
    """
    import pickle as pc
    from hyperopt import space_eval

    with open(path, "rb") as handle:
        trials = pc.load(handle)

    ok = [t for t in trials.trials
          if t.get("result", {}).get("status") == "ok"
          and t["result"].get("seconds") is not None]
    if not ok:
        raise RuntimeError(f"{path} holds no completed trial with a recorded "
                           f"duration; use prior sampling instead (drop --replay)")

    rng = np.random.default_rng(seed)
    pick = rng.choice(len(ok), size=min(n, len(ok)), replace=False)

    draws, local = [], []
    for i in pick:
        vals = {k: v[0] for k, v in ok[i]["misc"]["vals"].items() if v}
        draws.append(space_eval(space, vals))
        local.append(float(ok[i]["result"]["seconds"]))
    return draws, local, len(ok)


def own_setup(args):
    from epftoolbox.data import read_data

    begin, end = args.window
    H.assert_search_window_precedes_test(begin, end, args.dataset)
    dfTrain, dfTest = read_data(dataset=args.dataset, years_test=2,
                                path=args.datasets_dir,
                                begin_test_date=begin, end_test_date=end)
    n_exo = len(dfTrain.columns) - 1
    space = H.build_space(args.nlayers, 0, n_exo)
    kwargs = dict(nlayers=args.nlayers, dfTrain=dfTrain, dfTest=dfTest,
                  shuffle_train=1, dataset=args.dataset, data_augmentation=0,
                  calibration_window=args.calibration_years,
                  n_exogenous_inputs=n_exo)
    return H._objective, space, kwargs, {"n_exogenous": n_exo}


def multizone_setup(args):
    begin, end = args.window
    H.assert_search_window_precedes_test(begin, end, args.dataset)
    matrices = Z.load_zone_matrices(Z.ZONES, args.datasets_dir)

    out_zones = (args.zone,) if args.config == "wide" else Z.ZONES
    begin_n = pd.Timestamp(begin).normalize()
    end_n = pd.Timestamp(end).normalize()
    days = Z.available_days(matrices)
    test_days = days[(days >= begin_n) & (days <= end_n)]
    train_days = Z.training_days(matrices, begin_n, args.calibration_years)

    space = H.build_multizone_space(args.nlayers)
    kwargs = dict(nlayers=args.nlayers, matrices=matrices,
                  train_days=train_days, test_days=test_days, zones=Z.ZONES,
                  out_zones=out_zones)
    return (H._multizone_objective, space, kwargs,
            {"out_zones": list(out_zones), "n_train_days": int(len(train_days)),
             "n_test_days": int(len(test_days))})


def main():
    p = argparse.ArgumentParser(parents=[_pre])
    p.add_argument("--config", required=True, choices=["own", "wide", "joint"])
    p.add_argument("--zone", default="DK1")
    p.add_argument("--datasets-dir", default=os.path.join(HERE, "datasets"))
    p.add_argument("--draws", type=int, default=6)
    p.add_argument("--replay", default=None,
                   help="a completed trials pickle to replay evaluations from. "
                        "Pairs against the local seconds recorded in it, on the "
                        "search's own distribution of architectures.")
    p.add_argument("--draw-seed", type=int, default=20260904,
                   help="fixes WHICH architectures are timed; keep it identical "
                        "across devices or the comparison is not paired")
    p.add_argument("--nlayers", type=int, default=2)
    p.add_argument("--calibration-years", type=int, default=4)
    p.add_argument("--begin-test", default=str(Z.BEGIN_TEST.date()))
    p.add_argument("--out", default=None)
    p.add_argument("--label", default="")
    args = p.parse_args()

    begin_test = pd.Timestamp(args.begin_test)
    end = begin_test.normalize() - pd.Timedelta(hours=1)
    args.window = (end.normalize() - pd.Timedelta(days=HYPEROPT_DAYS), end)

    if args.config == "own":
        args.dataset = Z.dataset_name(args.zone)
        objective, space, kwargs, extra = own_setup(args)
    else:
        args.dataset = ("dnnjoint" if args.config == "joint"
                        else f"dnnwide_{args.zone}")
        objective, space, kwargs, extra = multizone_setup(args)

    gpus = tf.config.list_physical_devices("GPU")
    for g in gpus:                      # a search process should not seize 15 GB
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception:
            pass

    env = {
        "label": args.label,
        "config": args.config,
        "zone": args.zone,
        "dataset": args.dataset,
        "device_requested": args.device,
        "gpus_visible": [g.name for g in gpus],
        "gpu_details": ([tf.config.experimental.get_device_details(g) for g in gpus]
                        if gpus else []),
        "tensorflow": tf.__version__,
        "threads_requested": args.threads or None,
        "cpu_count": os.cpu_count(),
        "search_window": [str(args.window[0].date()), str(args.window[1])],
        "draw_seed": args.draw_seed,
        **extra,
    }
    print(json.dumps({"environment": env}, default=str), flush=True)

    if args.device == "gpu" and not gpus:
        print("!! asked for the GPU and TensorFlow sees none -- "
              "Runtime > Change runtime type > GPU, then Restart runtime.",
              flush=True)

    if args.replay:
        replayed, local_seconds, n_ok = _replay_draws(
            space, args.replay, args.draw_seed, args.draws)
        # A warm-up in front, which is not one of the replayed trials and is
        # never scored: it pays for CUDA context creation and the first graph
        # trace, which a 1000-evaluation search pays once.
        draws = _sample_draws(space, args.draw_seed, 1) + replayed
        local_seconds = [None] + local_seconds
        env["source"] = {"replay": os.path.basename(args.replay),
                         "completed_trials_in_file": n_ok}
    else:
        draws = _sample_draws(space, args.draw_seed, args.draws + 1)
        local_seconds = [None] * len(draws)
        env["source"] = {"prior_sampling": True}
    print(json.dumps({"source": env["source"]}), flush=True)

    scratch = os.path.join("/tmp" if os.path.isdir("/tmp") else HERE,
                           f"bench_trials_{args.config}_{args.device}")
    rows = []
    for i, hp in enumerate(draws):
        # Draw 0 is always the warm-up and is never scored.
        trials = Trials()
        _EPOCHS.clear()
        started = time.time()
        try:
            result = objective(hp, trials=trials, trials_file_path=scratch,
                               max_evals=args.draws, quiet=True, **kwargs)
            loss = float(result["loss"])
            failed = False
        except Exception as exc:                          # noqa: BLE001
            loss, failed = float("nan"), repr(exc)
        seconds = time.time() - started
        epochs = _EPOCHS[-1] if _EPOCHS else None

        neurons = [int(hp["neurons" + str(k)]) for k in range(1, args.nlayers + 1)
                   if int(hp["neurons" + str(k)]) >= 50]
        row = {
            "draw": i, "warmup": i == 0, "seconds": round(seconds, 3),
            "epochs": epochs,
            "seconds_per_epoch": round(seconds / epochs, 4) if epochs else None,
            "neurons": neurons, "activation": hp["activation"],
            "scaleX": hp["scaleX"], "scaleY": hp["scaleY"],
            "reg": hp["reg"]["val"], "batch_norm": bool(hp["batch_normalization"]),
            "loss": loss, "error": failed,
            "local_seconds": local_seconds[i],
            "speedup_vs_local": (round(local_seconds[i] / seconds, 2)
                                 if local_seconds[i] and seconds else None),
        }
        rows.append(row)
        print(json.dumps(row), flush=True)

    timed = [r for r in rows if not r["warmup"] and not r["error"]]
    secs = [r["seconds"] for r in timed]
    per_epoch = [r["seconds_per_epoch"] for r in timed if r["seconds_per_epoch"]]
    summary = {
        "summary": True, "config": args.config, "device": args.device,
        "label": args.label, "n_timed": len(timed),
        "mean_seconds": round(float(np.mean(secs)), 2) if secs else None,
        "median_seconds": round(float(np.median(secs)), 2) if secs else None,
        "total_seconds": round(float(np.sum(secs)), 1) if secs else None,
        "median_seconds_per_epoch": (round(float(np.median(per_epoch)), 4)
                                     if per_epoch else None),
        "mean_epochs": round(float(np.mean([r["epochs"] for r in timed])), 1) if timed else None,
        "warmup_seconds": rows[0]["seconds"],
    }
    paired = [r for r in timed if r["local_seconds"]]
    if paired:
        # Two ratios, because they answer different questions. The total ratio is
        # what a whole search costs (long evaluations dominate it, and they are
        # most of the wall time). The median ratio says whether the device helps
        # a typical evaluation, and is not moved by one outlier.
        summary["paired_n"] = len(paired)
        summary["local_total_seconds"] = round(sum(r["local_seconds"] for r in paired), 1)
        summary["total_speedup_vs_local"] = round(
            sum(r["local_seconds"] for r in paired)
            / sum(r["seconds"] for r in paired), 2)
        summary["median_speedup_vs_local"] = round(float(np.median(
            [r["local_seconds"] / r["seconds"] for r in paired])), 2)
    print(json.dumps(summary), flush=True)

    out = args.out or os.path.join(
        HERE, f"bench_{args.config}_{args.zone}_{args.device}.json")
    with open(out, "w") as fh:
        json.dump({"environment": env, "rows": rows, "summary": summary}, fh,
                  indent=2, default=str)
    print(f"wrote {out}", flush=True)


if __name__ == "__main__":
    main()


## 4. Run it

`DRAWS = 8` replays eight real trials per configuration per device, plus one
discarded warm-up. At the local rate that is roughly 4 minutes of work per
configuration; `joint` costs more.

The sample of trials is fixed by `--draw-seed`, so GPU and CPU replay the same
eight. Raise `DRAWS` if the per-trial ratios in the table scatter — the ratios are
much more stable than the absolute times, but eight is not many.

In [ ]:
DRAWS   = 8
CONFIGS = ['own', 'wide', 'joint']     # trim this while trying the notebook out

def run(config, device, draws=DRAWS, zone='DK1', threads=0, label=''):
    cmd = [sys.executable, 'bench_hyperopt_device.py',
           '--config', config, '--device', device, '--zone', zone,
           '--draws', str(draws), '--datasets-dir', DATASETS,
           '--replay', os.path.join(HYPER, TRIALS[config])]
    if threads:
        cmd += ['--threads', str(threads)]
    if label:
        cmd += ['--label', label]
    print('>', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=False)
    print()

for cfg in CONFIGS:
    run(cfg, 'gpu')

In [ ]:
# The same eight trials, the same process shape, GPU hidden. This is the control:
# it separates "the GPU is fast" from "Colab's machine is fast". Skip it if you
# only care about the headline number.
for cfg in CONFIGS:
    run(cfg, 'cpu')

## 5. Read the result

`total_speedup_vs_local` is the number the decision rule is about: local seconds
summed over the replayed trials, divided by Colab seconds over the same trials.
It weights long evaluations heavily, which is right — they are most of a search's
wall time. `median_speedup_vs_local` says whether a *typical* evaluation gets
faster and is not moved by one outlier. If the two disagree sharply, look at the
per-trial table: the GPU is helping wide architectures and not narrow ones.

In [ ]:
rows = []
for path in sorted(glob.glob('bench_*_*.json')):
    with open(path) as fh:
        blob = json.load(fh)
    s, e = blob['summary'], blob['environment']
    rows.append({'config': s['config'], 'device': s['device'], 'n': s['n_timed'],
                 'colab_s': s['mean_seconds'], 'local_s': s.get('local_total_seconds'),
                 'total_speedup': s.get('total_speedup_vs_local'),
                 'median_speedup': s.get('median_speedup_vs_local'),
                 's_per_epoch': s['median_seconds_per_epoch'],
                 'mean_epochs': s['mean_epochs'], 'warmup_s': s['warmup_seconds'],
                 'device_name': (e['gpu_details'][0].get('device_name')
                                 if e['gpu_details'] else f"CPU x{e['cpu_count']}")})
table = pd.DataFrame(rows).sort_values(['config', 'device'])
display(table.round(2))

In [ ]:
# Per replayed trial, so an average is not hiding a split between architectures.
for path in sorted(glob.glob('bench_*_gpu.json')):
    with open(path) as fh:
        blob = json.load(fh)
    df = pd.DataFrame([r for r in blob['rows'] if not r['warmup']])
    if df.empty or 'local_seconds' not in df:
        continue
    print(blob['summary']['config'])
    display(df[['neurons', 'activation', 'batch_norm', 'epochs',
                'local_seconds', 'seconds', 'speedup_vs_local']].round(2))

In [ ]:
# What each configuration's whole E=1000 search would cost at the measured rate,
# and therefore what moving it off the laptop actually buys.
E = 1000
COUNT  = {'own': 7, 'wide': 2, 'joint': 2}                   # joint-DK + joint-7
LOCAL  = {'own': 25.0, 'wide': 26.8, 'joint': 33.1}          # handover section 9

total_local = total_gpu = 0.0
for _, r in table[table.device == 'gpu'].iterrows():
    n = COUNT.get(r.config, 1)
    local_h = E * LOCAL[r.config] * n / 3600
    gpu_h   = local_h / r.total_speedup if r.total_speedup else float('nan')
    total_local += local_h; total_gpu += gpu_h
    print(f'{r.config:6s} x{n}: local {local_h:6.1f} core-h -> Colab {gpu_h:6.1f} h '
          f'({r.total_speedup:.2f}x), saves {local_h - gpu_h:6.1f} h')
print(f'\nsearch total: {total_local:.1f} core-h local -> {total_gpu:.1f} h on Colab')
print(f'the backtest stays local either way: 53.4 core-h')

## 6. Deciding

- **5× or better** → move the searches. The laptop then does only the ~53
  backtest core-hours, and each search hands back one small trials pickle that
  `run_dnn_dk1.py --skip-hyperopt` picks up unchanged.
- **around 1.2×** → drop it. The saving does not pay for three trials files
  crossing machines, Colab's session limits, and a second environment to keep
  identical.
- **in between** → note that only `joint` and `wide` need to move to shorten the
  critical path; `own ×7` is 48.6 h of embarrassingly parallel work that the
  8-core laptop absorbs alongside them.

**Before trusting a large number**, check `mean_epochs` on the GPU against the CPU
run for the same trials. If the GPU ran far fewer epochs it is early-stopping
sooner, not computing faster, and `s_per_epoch` is the column to compare.

**If the searches do move**, the invariant that matters is §3.4 of the handover:
E and S identical across every run. Moving *some* searches to a GPU does not
threaten that — E is a count, not a duration — but a search abandoned halfway
because a Colab session was reclaimed does. `_objective` checkpoints the trials
file before every evaluation, so a killed session loses at most one evaluation;
re-running with the same trials file resumes. Check `len(trials.trials) == 1000`
before using one.